# Gated CAE — Evaluation on CelebA
Drive 마운트 → 경로 설정 → evaluate.py 실행 → visualization.py로 시각화

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정
> `PROJECT_DIR` 와 `CKPT_PATH` 를 본인 Drive 경로에 맞게 수정하세요.

In [ ]:
import sys, os

# ✏️ 수정 필요
PROJECT_DIR = '/content/mask-aware-inpainting'
CKPT_PATH   = f'{PROJECT_DIR}/checkpoints/epoch020.pth'

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Project   :', PROJECT_DIR)
print('Checkpoint:', CKPT_PATH)
print('Exists    :', os.path.exists(CKPT_PATH))

## 3. evaluate.py 실행
`evaluate.py` 의 `evaluate()` 함수를 그대로 호출합니다.

In [ ]:
from evaluate import evaluate

# evaluate.py 내부에서 PSNR/SSIM 출력 + eval_gated_sample.png 저장
evaluate(CKPT_PATH)

## 4. visualization.py로 저장된 그리드 이미지 확인
`evaluate()` 가 내부적으로 `save_grid()` 를 호출해 저장한 PNG를 불러와 표시합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img_path = f'{PROJECT_DIR}/eval_gated_sample.png'

img = mpimg.imread(img_path)
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Masked Input  |  [GatedCAE Output]  |  Ground Truth', fontsize=12) #각자 모델에 맞게 수정
plt.tight_layout()
plt.show()

## 5. 추가 샘플 시각화 (save_grid 직접 호출)
원하는 이미지 인덱스를 골라 `save_grid()` 로 별도 그리드를 생성합니다.

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from config import Config
from datasets.celeba_dataset import CelebAInpaintingDataset
from models.gated_cae import GatedCAE
from utils.visualization import save_grid  # visualization.py 직접 import

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg    = Config()

dataset = CelebAInpaintingDataset(root=cfg.data_root, split='test',
                                  image_size=cfg.image_size)

model = GatedCAE(base_ch=cfg.base_ch).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# ✏️ 보고 싶은 인덱스 변경 가능
indices = [0, 5, 10, 20]

masked_list, output_list, gt_list = [], [], []
with torch.no_grad():
    for idx in indices:
        masked, mask, gt = dataset[idx]
        out = model(masked.unsqueeze(0).to(device),
                    mask.unsqueeze(0).to(device))[0].cpu()
        masked_list.append(masked)
        output_list.append(out)
        gt_list.append(gt)

masked_t = torch.stack(masked_list)
output_t = torch.stack(output_list)
gt_t     = torch.stack(gt_list)

out_path = f'{PROJECT_DIR}/eval_gated_extra.png'
save_grid(masked_t, output_t, gt_t, out_path, nrow=len(indices))

img = mpimg.imread(out_path)
plt.figure(figsize=(14, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Masked  |  Output  |  GT  (추가 샘플)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. 학습 곡선 (PSNR / SSIM)

In [ ]:
import csv
import matplotlib.pyplot as plt

CSV_PATH = f'{PROJECT_DIR}/analysis_results/model_performance_metrics.csv'

epochs, psnrs, ssims = [], [], []
with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row['epoch']))
        psnrs.append(float(row['psnr']))
        ssims.append(float(row['ssim']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, psnrs, marker='o', color='steelblue')
ax1.set_title('Epoch별 PSNR 변화')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('PSNR (dB)')
ax1.grid(True)

ax2.plot(epochs, ssims, marker='o', color='goldenrod')
ax2.set_title('Epoch별 SSIM 변화')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('SSIM')
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f'최종 PSNR: {psnrs[-1]:.2f} dB  |  최종 SSIM: {ssims[-1]:.4f}')